In [76]:
from rich import print as pprint

# O que é metaprogramação

Metaprogramação é a programação de programas que escrevem ou manipulam outros programas (ou a si próprios) assim como seus dados, ou que fazem parte do trabalho em tempo de compilação. Em alguns casos, isso permite que os programadores sejam mais produtivos ao evitar que parte do código seja escrita manualmente.

## Exemplo em bash

```bash
#!/bin/bash
# metaprograma - Esse código gera um novo programa com 993 linhas que imprime os números da faixa 1–992
echo '#!/bin/bash' > programa
for ((I=1; I<=992; I++)) do
    echo "echo $I" >> programa
done
chmod +x programa
```

Em python, ha varios jeitos de se fazer metaprogramacao. 

| TECHNIQUE              | DESCRIPTION                                          |
|------------------------|------------------------------------------------------|
| Metaclasses            | Classes that create/modify other classes. Control class creation process. |
| Decorators             | Functions that modify/extend other functions/methods without changing source. |
| AST (Abstract Syntax Trees) | Parse code into tree structures for analysis/modification before execution. |
| Descriptors            | Essentially allow us to modify the behavior of the dot (`.`) operator. Objects controlling attribute access via `__get__`, `__set__`, `__delete__`. |
| Dynamic Attributes     | Control attribute access using `__getattr__`, `__setattr__`, etc. |
| Dynamic Execution      | Execute code dynamically using `eval()`, `exec()`, `compile()`. |
| Monkey Patching        | Modify classes/modules at runtime. |
| Class Decorators       | Modify/enhance classes during definition. |
| `__init_subclass__`    | Hook method called when subclass is created (alternative to metaclasses). |
| Type Hints/Annotations | Use type annotations for runtime introspection and metaprogramming. |
| Context Managers       | Define resource management using `__enter__` / `__exit__`. |
| Dynamic Imports        | Import modules dynamically using `importlib`. |
| Function Attributes    | Attach metadata to functions. |
| `__new__` Method       | Control instance creation (before `__init__`). |

# Decoradores

*Decorators* (decoradores) são uma forma elegante de envolver uma função, método ou classe com outro código, adicionando comportamento antes/depois (ou até substituindo) sem alterar a implementação original.

In [77]:
# transfer all fn characteristics to the decorated function
from functools import wraps

In [78]:
def debugger(fn):
    @wraps(fn)   # transfer all fn characteristics to the decorated function
    def inner(*args, **kwargs):
        pprint(f'decorating function {fn.__qualname__}', args, kwargs)
        return fn(*args, **kwargs)
    return inner

## How to use

In [79]:
def func_0(*args, **kwargs):
    print("Executing code in func_0")

In [80]:
func_0 = debugger(func_0)
func_0()

decorating function func_0
()
{}

Executing code in func_0


## Sugar/pie syntax

In [ ]:
@debugger
def func_1(*args, **kwargs):
    pass

@debugger
def func_2(*args, **kwargs):
    pass

## Para que servem?

- Logging/tracing de chamadas  
- Cache/memoization (ex.: functools.lru_cache)  
- Validação/autorização antes de executar  
- Reintentos com backoff  
- Métrica/tempo de execução  
- Conversão de funções em propriedades/métodos especiais (```@property```, ```@staticmethod```, ```@classmethod```)

## Ordem de aplicação

```python
@A
@B
def f():
    ...
# Equivale a: f = A(B(f))
```

## Decoradores parametrizados

São decoradores que aceitam argumentos. Como a sintaxe  
`@decorador(...)` chama o decorador na hora da definição, precisamos de uma fábrica de decoradores: uma função que recebe os parâmetros e retorna o decorador real.

Decoradores parametrizados funcionam em 3 camadas:

1. Fábrica (recebe parâmetros do usuário)  
2. Decorador (recebe a função)  
3. Wrapper (executa antes/depois e chama a função)

In [19]:
# Exemplo: retry com parâmetros
import functools, time
def retry(times=3, exceptions=(Exception,), delay=0.0): # fabrica de decoradores, recebendo parametros do usuario
    def decorator(func):  # decorador propriamente dito, recebendo a funcao a ser decorada como argumento
        @functools.wraps(func)
        def wrapper(*args, **kwargs): # wrapper, tb comumente chamado de inner. repassa os argumentos e kw argumentos para funcao
            last = None
            for i in range(times):
                print(f"try #{i}")
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last = e
                    if i < times - 1 and delay:
                        time.sleep(delay)
            raise last
        return wrapper
    return decorator


In [104]:
def failed_function_0(times=5):
    for _ in range(times):
        raise Exception("I have failed, sorry..")

In [105]:
decorated_failed_fn = retry(times=5, delay=0.5)(failed_function_0)

In [106]:
decorated_failed_fn()

try #0
try #1
try #2
try #3
try #4


Exception: I have failed, sorry..

In [20]:
@retry(times=5, delay=0.5)
def failed_function(times=5):
    for _ in range(times):
        raise Exception("I have failed, sorry..")

In [21]:
failed_function()

try #0
try #1
try #2
try #3
try #4


Exception: I have failed, sorry..

## Exemplos MLOps

### Logger MLOps

In [ ]:
from functools import wraps
from time import perf_counter
from typing import Callable, Optional, Dict, Any, Union
import logging
import traceback
import sys


In [ ]:
class CustomLogger:
    COLOR_RESET = "\033[0m"
    COLOR_INFO = "\033[32m"     # Green
    COLOR_WARNING = "\033[93m"  # Yellow
    COLOR_ERROR = "\033[91m"    # Red
    COLOR_EXTRA = "\033[94m"    # Blue

    logger_format_extras = {
        "INFO": f"{COLOR_INFO}{{asctime}} | {{levelname:<8}} | {{message}} | \nextra info: {COLOR_EXTRA}{{custom_dimensions}}{COLOR_RESET}",
        "WARNING": f"{COLOR_WARNING}{{asctime}} | {{levelname:<8}} | {{message}} | \nextra info: {COLOR_EXTRA}{{custom_dimensions}}{COLOR_RESET}",
        "ERROR": f"{COLOR_ERROR}{{asctime}} | {{levelname:<8}} | {{message}} | \nextra info: {COLOR_EXTRA}{{custom_dimensions}}{COLOR_RESET}",
    }

    logger_format = {
        "INFO": f"{COLOR_INFO}{{asctime}} | {{levelname:<8}} | {{message}}{COLOR_RESET}",
        "WARNING": f"{COLOR_WARNING}{{asctime}} | {{levelname:<8}} | {{message}}{COLOR_RESET}",
        "ERROR": f"{COLOR_ERROR}{{asctime}} | {{levelname:<8}} | {{message}}{COLOR_RESET}",
    }

    def __init__(self, stage: str = "dev"):
        self.stage = stage
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)
        self.clear_logger()

    def clear_logger(self):
        if self.logger.hasHandlers():
            self.logger.handlers.clear()

    def _configure_stage(self, extra: Dict[str, Any]):
        if "stage" not in extra:
            return

        stage = extra.get("stage", "prod").lower()

        if stage in ("prod", "production") and self.stage != "prod":
            self.stage = "prod"
        elif stage in ("dev", "development", "test", "testing", "staging", "stage") and self.stage != "dev":
            self.stage = "dev"

    def _configure_extras(self, extra) -> Dict:
        self._configure_stage(extra)
        return {"custom_dimensions": extra}

    def warning(self, message: str, extra: Optional[Dict[str, Any]] = None) -> None:
        self.clear_logger()
        handler = logging.StreamHandler(sys.stdout)
        if extra not in [{}, None]:
            formatter = logging.Formatter(
                self.logger_format_extras["WARNING"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            properties = self._configure_extras(extra)
            self.logger.warning(message, extra=properties)
        else:
            formatter = logging.Formatter(
                self.logger_format["WARNING"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            self.logger.warning(message)

    def info(self, message: str, extra: Optional[Dict[str, Any]] = None) -> None:
        self.clear_logger()
        handler = logging.StreamHandler(sys.stdout)
        if extra not in [{}, None]:
            formatter = logging.Formatter(
                self.logger_format_extras["INFO"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            properties = self._configure_extras(extra)
            self.logger.info(message, extra=properties)
        else:
            formatter = logging.Formatter(
                self.logger_format["INFO"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            self.logger.info(message)

    def error(
        self, message: Union[str, Exception], extra: Optional[Dict[str, Any]] = None
    ) -> None:
        self.clear_logger()
        if isinstance(message, Exception):
            message = str(message)
        handler = logging.StreamHandler(sys.stderr)
        if extra not in [{}, None]:
            formatter = logging.Formatter(
                self.logger_format_extras["ERROR"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            properties = self._configure_extras(extra)
            self.logger.error(message, extra=properties, exc_info=True)
        else:
            formatter = logging.Formatter(
                self.logger_format["ERROR"], style='{', datefmt='%Y-%m-%d %H:%M:%S'
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
            self.logger.error(message, exc_info=True)





In [ ]:
def log_with_logger(logger_instance, log_level, message, extra=None):
    logger_instance.clear_logger()
    handler = logging.StreamHandler(sys.stdout)
    if extra not in [{}, None]:
        formatter = logging.Formatter(
            logger_instance.logger_format_extras[log_level], style='{', datefmt='%Y-%m-%d %H:%M:%S'
        )
        handler.setFormatter(formatter)
        logger_instance.logger.addHandler(handler)
        properties = logger_instance._configure_extras(extra)
        getattr(logger_instance.logger, log_level.lower())(message, extra=properties)
    else:
        formatter = logging.Formatter(
            logger_instance.logger_format[log_level], style='{', datefmt='%Y-%m-%d %H:%M:%S'
        )
        handler.setFormatter(formatter)
        logger_instance.logger.addHandler(handler)
        getattr(logger_instance.logger, log_level.lower())(message)

In [ ]:
def logger_decorator(extra: Optional[Dict[str, Any]] = None):
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            logger_instance = CustomLogger()
            try:
                logger_instance.info(f"Calling {func.__name__}", extra)
                start = perf_counter()
                result = func(*args, **kwargs)
                end = perf_counter()
            except Exception as excpt:
                exception_data = {
                    type(excpt).__name__: str(excpt),
                    "traceback": traceback.format_exc(),
                }
                logger_instance.error(f"{func.__name__} failed", exception_data)
                raise excpt
            else:
                logger_instance.info(f"{func.__name__} elapsed time {(end-start):.4f}s")
                return result
        return wrapper
    return decorator

#### Exemplos/Como usar

In [ ]:
logger = CustomLogger()
logger.info(f"My func is executing.")
logger.warning(f"My func is burning.")
logger.error(f"My func is caput.")

# Decorated logger
@logger_decorator(extra={"stage": "prod"})
def success_function():
    print("Inside the decorated function")


@logger_decorator(extra={"stage": "prod"})
def failed_function():
    1 / 0
    print("Inside the decorated function")


success_function()
failed_function()

### Old but Gold - Telemetria Melchior

<p align="center">
<img src="imgs/logo.png" alt="Telemetria Melchior" width="200">
</p>

Decoradores sao uma forma poderosa de se adicionar funcoes a outras funcoes sem alterar o codigo. 
A telemetria nao e so uma funcao que decora outra(s). e uma arquitetura inteira por detras dos decoracores. 

<p align="center">
<img src="imgs/croqui.png" alt="Croqui da aplicação" width="600">
</p>

<p align="center">
<img src="imgs/software_structure.png" alt="Scientia vinces" width="600">
</p>

## Decoradores de classes

Decoradores geralmente sao pensados em funcoes que decoram outras funcoes. 
Python é nosso amigo, cidadão. Assim sendo, podemos ter decoradores que decoram classes.

In [22]:
# Exemplo: decoradores que adicionam atributos a uma classe
def savings(cls):
    cls.account_type = 'savings'
    return cls
    
def checking(cls):
    cls.account_type = 'checking'
    return cls

In [23]:
class Account:
    pass

@savings
class Bank1Savings(Account):
    pass

@savings
class Bank2Savings(Account):
    pass

@checking
class Bank1Checking(Account):
    pass

@checking
class Bank2Checking(Account):
    pass

In [24]:
Bank1Savings.account_type, Bank1Checking.account_type

('savings', 'checking')

### DRY mode:

In [26]:
def account_type(type_):
    def decorator(cls):
        cls.account_type = type_
        return cls
    return decorator

In [27]:
@account_type('Savings')
class Bank1Savings:
    pass

@account_type('Checking')
class Bank1Checking:
    pass

In [28]:
Bank1Savings.account_type, Bank1Checking.account_type

('Savings', 'Checking')

In [29]:
# exemplo: outras formas 

In [30]:
def hello(cls):
    cls.hello = lambda self: f'{self} says hello!'
    return cls

In [31]:
@hello
class Person:
    def __init__(self, name):
        self.name = name
        
    def __str__(self):
        return self.name

In [32]:
vars(Person)

mappingproxy({'__module__': '__main__',
              '__init__': <function __main__.Person.__init__(self, name)>,
              '__str__': <function __main__.Person.__str__(self)>,
              '__dict__': <attribute '__dict__' of 'Person' objects>,
              '__weakref__': <attribute '__weakref__' of 'Person' objects>,
              '__doc__': None,
              'hello': <function __main__.hello.<locals>.<lambda>(self)>})

In [33]:
p = Person('Helo Mundo')
p.hello()

'Helo Mundo says hello!'

### Problema: como fazer um logger para cada metodo de uma classe

##### Decorando cada metodo individualmente

In [81]:
from functools import wraps

def func_logger(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        result = fn(*args, **kwargs)
        pprint(f'log: {fn.__qualname__}({args}, {kwargs}) = {result}')
        return result
    return inner    

In [82]:
class Person:
    @func_logger
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    @func_logger
    def greet(self):
        return f'Hello, my name is {self.name} and I am {self.age}'

In [83]:
p = Person('John', 78)
p.greet()

log: Person.__init__((<__main__.Person object at 0x000002B61B9DF9D0>, 'John', 78), {}) = None

log: Person.greet((<__main__.Person object at 0x000002B61B9DF9D0>,), {}) = Hello, my name is John and I am 78

'Hello, my name is John and I am 78'

##### Automatizando: Verificar, metodo a metodo, se ele e um objeto do tipo *callable*

In [84]:
def class_logger(cls):
    for name, obj in vars(cls).items():
        if callable(obj):
            pprint('decorating:', cls, name)
            setattr(cls, name, func_logger(obj))
    return cls

In [85]:
@class_logger
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def greet(self):
        return f'Hello, my name is {self.name} and I am {self.age}'

decorating: <class '__main__.Person'> __init__

decorating: <class '__main__.Person'> greet

In [86]:
pprint(vars(Person))

mappingproxy({
    '__module__': '__main__',
    '__init__': <function Person.__init__ at 0x000002B61B2132E0>,
    'greet': <function Person.greet at 0x000002B61B99E8C0>,
    '__dict__': <attribute '__dict__' of 'Person' objects>,
    '__weakref__': <attribute '__weakref__' of 'Person' objects>,
    '__doc__': None
})

`mappingproxy` é uma visão somente-leitura do dicionário de atributos da classe.

Quando você faz `vars(Person)` (ou `Person.__dict__`), o que recebe não é um `dict` normal, mas um wrapper imutável chamado **mapping proxy** (tipo `types.MappingProxyType`). Ele mostra o conteúdo atual da namespace da classe, mas impede que você altere direto por ali.

**O que isso significa na prática**

- **Visualização imutável**: você pode ler como se fosse um `dict`, mas não pode fazer `mappingproxy['x'] = 1` (vai dar `TypeError`).
- **Reflete mudanças reais na classe**: se você alterar a classe de forma normal, o `mappingproxy` "se atualiza", porque ele é uma janela para o dicionário interno da classe.
- **Instâncias são diferentes**: `vars(obj)` para um objeto normalmente retorna um `dict` mutável (o `__dict__` da instância). Para classes, retorna esse `mappingproxy`.

In [55]:
d = vars(Person)          # mappingproxy(...)
print(type(d))            # <class 'mappingproxy'>

<class 'mappingproxy'>


In [87]:
try:
    d['nova_attr'] = 123
except TypeError as e:
    pprint(f"falha {e}")

falha 'mappingproxy' object does not support item assignment

In [88]:
p = Person('John', 78)
p.greet()

log: Person.__init__((<__main__.Person object at 0x000002B61B9DD0C0>, 'John', 78), {}) = None

log: Person.greet((<__main__.Person object at 0x000002B61B9DD0C0>,), {}) = Hello, my name is John and I am 78

'Hello, my name is John and I am 78'

Na forma que está, o decorador nao vai funcionar com ```@staticmethod``` e ```@classmethod``` 
porque os metodos decorados por eles deixam de ser callables e passam a ser descritores. 



Explicação: `@staticmethod` e `@classmethod` são **descritores não-dados (non-data descriptors)**.

Isso significa que eles implementam só `__get__` (não têm `__set__`/`__delete__`). Consequências:

**Precedência de lookup:**
_data descriptors >_ `instância.__dict__` > _non-data descriptors >_ demais atributos da classe.

Então, um atributo na instância pode ofuscar (shadow) um `@staticmethod`/`@classmethod`.

**Como cada um se comporta no `__get__`:**

- `staticmethod`: devolve a função crua, sem binding de `self`/`cls`  
  (é basicamente `return self.__func__`).

- `classmethod`: devolve um callable que injeta `cls` (a classe) como primeiro argumento  
  (equivale a retornar algo como `lambda *a, **k: self.__func__(owner, *a, **k)`).

**Comparação com @property:** `property` é data descriptor (tem `__set__`/`__delete__`), por isso vence o `__dict__` da instância — já `staticmethod`/`classmethod` não.

```@property``` transforma um método em um atributo calculados/validados. 

Implementa ```__get__```, ```__set__``` e ```__delete__``` (é data descriptor), então tem precedência sobre ```__dict__``` da instância. 

entao, o decorador ```class_logger``` nao vai conseguir funcionar corretamente. 

para fazer isso, ele precisa identificar esses objetos, desempacota-los para pegar a funcao original, logar essa funcao e reempacotar com o @original. 

In [90]:
class X:
    @staticmethod
    def s(): 
        return "sou static"
    
    @classmethod
    def c(cls): 
        return f"sou class de {cls.__name__}"
    
    def method(self):
        pass

pprint(X.__dict__['method'])      # funcao/metodo simples
pprint(X.__dict__['s'])           # <staticmethod objeto>
pprint(X.__dict__['s'].__func__)  # função original
pprint(X.__dict__['c'])         # <classmethod objeto>
pprint(X.__dict__['c'].__func__)  # função original que espera cls

<function X.method at 0x000002B61B99E050>

<staticmethod(<function X.s at 0x000002B61B99E9E0>)>

<function X.s at 0x000002B61B99E9E0>

<classmethod(<function X.c at 0x000002B61B99EB90>)>

<function X.c at 0x000002B61B99EB90>

In [91]:
def class_logger(cls):
    for name, obj in vars(cls).items():
        if callable(obj):
            pprint('decorating:', cls, name)
            setattr(cls, name, func_logger(obj))
        elif isinstance(obj, staticmethod):
            original_func = obj.__func__
            pprint('decorating static method', original_func)
            decorated_func = func_logger(original_func)
            method = staticmethod(decorated_func)
            pprint(method, type(method))
            setattr(cls, name, method)
        elif isinstance(obj, classmethod):
            original_func = obj.__func__
            pprint('decorating class method', original_func)
            decorated_func = func_logger(original_func)
            method = classmethod(decorated_func)
            setattr(cls, name, method)
        elif isinstance(obj, property):
            pprint('decorating property', obj)
            if obj.fget:
                obj = obj.getter(func_logger(obj.fget))
            if obj.fset:
                obj = obj.setter(func_logger(obj.fset))
            if obj.fdel:
                obj = obj.deleter(func_logger(obj.fdel))
            setattr(cls, name, obj)
    return cls

In [92]:
@class_logger
class Person:
    def __init__(self, name):
        self._name = name
        
    @property
    def name(self):
        return self._name
    
    @name.setter
    def name(self, value):
        self._name = value
        
    @name.deleter
    def name(self):
        print('deleting name...')

decorating: <class '__main__.Person'> __init__

decorating property <property object at 0x000002B61B9CF1A0>

In [93]:
p = Person('David')

log: Person.__init__((<__main__.Person object at 0x000002B61B3E8160>, 'David'), {}) = None

In [94]:
p.name

log: Person.name((<__main__.Person object at 0x000002B61B3E8160>,), {}) = David

'David'

In [95]:
p.name = 'Beazley'

log: Person.name((<__main__.Person object at 0x000002B61B3E8160>, 'Beazley'), {}) = None

In [96]:
del p.name

deleting name...


log: Person.name((<__main__.Person object at 0x000002B61B3E8160>,), {}) = None

#### DRY mode: usando ```inspect```

In [47]:
import inspect

In [97]:
def class_logger(cls):
    for name, obj in vars(cls).items():
        if isinstance(obj, staticmethod) or isinstance(obj, classmethod):
            type_ = type(obj)
            original_func = obj.__func__
            pprint(f'decorating {type_.__name__} method', original_func)
            decorated_func = func_logger(original_func)
            method = type_(decorated_func)
            setattr(cls, name, method)
        elif isinstance(obj, property):
            pprint('decorating property', obj)
            methods = (('fget', 'getter'), ('fset', 'setter'), ('fdel', 'deleter'))
            for prop, method in methods:
                if getattr(obj, prop):
                    obj = getattr(obj, method)(func_logger(getattr(obj, prop)))
            setattr(cls, name, obj)
        elif inspect.isroutine(obj):
            pprint('decorating:', cls, name)
            setattr(cls, name, func_logger(obj))
    return cls

In [98]:
@class_logger
class MyClass:
    @staticmethod
    def static_method():
        print('static_method called...')
    
    @classmethod
    def cls_method(cls):
        print('class method called...')
    
    def inst_method(self):
        print('instance method called...')
    
    @property
    def name(self):
        print('name getter called...')
        
    @name.setter
    def name(self, value):
        print('name setter called...')
        
    @name.deleter
    def name(self):
        print('name deleter called...')
    
    def __add__(self, other):
        print('__add__ called...')
    
    @class_logger
    class Other:
        def __call__(self):
            print(f'{self}.__call__ called...')
        
    other = Other()

decorating: <class '__main__.MyClass.Other'> __call__

decorating staticmethod method <function MyClass.static_method at 0x000002B61B99F370>

decorating classmethod method <function MyClass.cls_method at 0x000002B61B99F0A0>

decorating: <class '__main__.MyClass'> inst_method

decorating property <property object at 0x000002B61B9F8D10>

decorating: <class '__main__.MyClass'> __add__

In [99]:
MyClass().name

name getter called...


log: MyClass.name((<__main__.MyClass object at 0x000002B61B973670>,), {}) = None

In [100]:
MyClass().name = 'David'

name setter called...


log: MyClass.name((<__main__.MyClass object at 0x000002B61B972C50>, 'David'), {}) = None

In [101]:
del MyClass().name

name deleter called...


log: MyClass.name((<__main__.MyClass object at 0x000002B61B9731F0>,), {}) = None

#### <font color=red>Atenção: codigos 100% DRY podem nao ser legiveis o suficiente. de repente a repeticao deixa a leitura facilitada</font>

```python
def class_logger(cls):
    for name, obj in vars(cls).items():
        if isinstance(obj, staticmethod) or isinstance(obj, classmethod):
            type_ = type(obj)
            original_func = obj.__func__
            print(f'decorating {type_.__name__} method', original_func)
            decorated_func = func_logger(original_func)
            method = type_(decorated_func)
            setattr(cls, name, method)
        elif isinstance(obj, property):
            # deste modo, embora seja repetitivo, 
            # a legibilidade e interpretabilidade do codigo e melhor
            print('decorating property', obj)
            if obj.fget:
                obj = obj.getter(func_logger(obj.fget))
            if obj.fset:
                obj = obj.setter(func_logger(obj.fset))
            if obj.fdel:
                obj = obj.deleter(func_logger(obj.fdel))
            setattr(cls, name, obj)
        elif inspect.isroutine(obj):
            print('decorating:', cls, name)
            setattr(cls, name, func_logger(obj))
    return cls
```

## Classes decoradoras

In [57]:
# fn capturada pela closure
# fn e chamada atrasves de wrapper
def decorator(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper

In [103]:
import types

def func():
    pass
decorated_func = decorator(func)
# decorated_func e uma funcao
# funcoes nao sao descritores de dados. python chama o metodo __get__ de uma funcao
pprint(
    f"type(decorated_func): {type(decorated_func)}, \n"
    f"isinstance(decorated_func, types.FunctionType): {isinstance(decorated_func, types.FunctionType)}"
)

type(decorated_func): <class 'function'>, 
isinstance(decorated_func, types.FunctionType): True

In [74]:
# funcoes nao sao descritores de dados. python chama o metodo __get__ de uma funcao

from rich.console import Console
from rich.columns import Columns
from rich.text import Text

console = Console()

# For regular functions
def my_function(a, b):
    return a + b

attr_list = [attr for attr in dir(my_function)]
styled_list = [Text(attr_item, style="red") if attr_item == "__get__" else Text(attr_item, style="blue") for attr_item in attr_list]

columns = Columns(styled_list)
console.print(columns)

__annotations__  __builtins__     __call__      __class__  __closure__ __code__   __defaults__     
__delattr__      __dict__         __dir__       __doc__    __eq__      __format__ __ge__           
__get__          __getattribute__ __globals__   __gt__     __hash__    __init__   __init_subclass__
__kwdefaults__   __le__           __lt__        __module__ __name__    __ne__     __new__          
__qualname__     __reduce__       __reduce_ex__ __repr__   __setattr__ __sizeof__ __str__          
__subclasshook__

In [75]:
# fn capturada pelo __init__
# classes decoradoras sao callables e precisam implementar o metodo __call__
# fn e chamada atrasves da chamada de __call__
class Decorator:
    def __init__(self, fn):
        self.fn = fn
    def __call__(cls, *args, **kwargs):
        return self.fn(*args, **kwargs)

class_decorated_func = Decorator(func)
# class_decorated_func e uma instancia da classe Decorator
pprint(
    f"type(class_decorated_func): {type(class_decorated_func)}, \n"
    f"isinstance(class_decorated_func, Decorator): {isinstance(class_decorated_func, Decorator)}"
)

type(class_decorated_func): <class '__main__.Decorator'>, 
isinstance(class_decorated_func, Decorator): True

## Class decorator in a method

In [107]:
class Logger:
    def __init__(self, fn):
        self.fn = fn
        
    def __call__(self, *args, **kwargs):
        print(f'Log: {self.fn.__name__} called.')
        return self.fn(*args, **kwargs)

In [108]:
# using in a function
@Logger
def say_hello():
    pass

say_hello()

Log: say_hello called.


In [109]:
pprint(type(say_hello))

<class '__main__.Logger'>

In [110]:
# using in a method
class Person:
    def __init__(self, name):
        self.name = name
        
    @Logger
    def say_hello(self):
        return f'{self.name} says hello!'

In [111]:
p = Person('David')

In [112]:
p.say_hello()

Log: say_hello called.


TypeError: Person.say_hello() missing 1 required positional argument: 'self'

Por que python esta reclamando que `self` nao esta sendo passado para o metodo `say_hello`?

Estamos a chamar o metodo a partir da instancia, onde o `self` nao esta sendo passado para o metodo. 

Lembremos que `say_hello` esta sendo decorado por uma classe decoradora. Assim sendo, essa forma de decoracao devolve uma instancia da classe decoradora `Logger`. 

Entao, o metodo `say_hello` deixou de ser um metodo ligado a classe `Person` e passou a ser um objeto. 

Funcoes implementam o metodo __get__que tambem e usado para criar um metodo ligado a classe (*bound method*)

Nossa classe decoradora `Logger` nao implementa o metodo `__get__`, que e usado para criar um *bound method* na classe Person, no caso. 

In [113]:
pprint(type(p.say_hello))

<class '__main__.Logger'>

In [115]:
# analisando os metodos implementados para say_hello:
attr_list = [attr for attr in dir(p.say_hello)]
styled_list = [Text(attr_item, style="red") if attr_item == "__get__" else Text(attr_item, style="blue") for attr_item in attr_list]
columns = Columns(styled_list)
console.print(columns)

__call__   __class__        __delattr__ __dict__   __dir__       __doc__           __eq__      __format__
__ge__     __getattribute__ __gt__      __hash__   __init__      __init_subclass__ __le__      __lt__    
__module__ __ne__           __new__     __reduce__ __reduce_ex__ __repr__          __setattr__ __sizeof__
__str__    __subclasshook__ __weakref__ fn        

In [116]:
# ajustes: Implementar __get__ na classe decoradora Logger para deixar td funcionando como padrao
from types import MethodType

class Logger:
    def __init__(self, fn):
        self.fn = fn
        
    def __call__(self, *args, **kwargs):
        print(f'Log: {self.fn.__name__} called.')
        return self.fn(*args, **kwargs)
    
    def __get__(self, instance, owner_class):
        print(f'__get__ called: self={self}, instance={instance}')
        if instance is None:
            print('\treturning self unbound...')
            return self
        else:
            # self is callable, since it implements __call__
            print('\treturning self as a method bound to instance')
            return MethodType(self, instance)

In [117]:
class Person:
    def __init__(self, name):
        self.name = name
        
    @Logger
    def say_hello(self):
        return f'{self.name} says hello!'

In [118]:
p = Person('David')
p.say_hello()

__get__ called: self=<__main__.Logger object at 0x000002B61BD548B0>, instance=<__main__.Person object at 0x000002B61BD54370>
	returning self as a method bound to instance
Log: say_hello called.


'David says hello!'

In [119]:
# reanalisando os metodos implementados para say_hello:
attr_list = [attr for attr in dir(p.say_hello)]
styled_list = [Text(attr_item, style="red") if attr_item == "__get__" else Text(attr_item, style="blue") for attr_item in attr_list]
columns = Columns(styled_list)
console.print(columns)

__get__ called: self=<__main__.Logger object at 0x000002B61BD548B0>, instance=<__main__.Person object at 0x000002B61BD54370>
	returning self as a method bound to instance


__call__ __class__        __delattr__   __dir__  __doc__  __eq__            __format__ __func__ __ge__          
__get__  __getattribute__ __gt__        __hash__ __init__ __init_subclass__ __le__     __lt__   __ne__          
__new__  __reduce__       __reduce_ex__ __repr__ __self__ __setattr__       __sizeof__ __str__  __subclasshook__
fn      

Agora, o metodo `say_hello` é um *boud method*. E ligou o *callable* da instancia de `Logger`com a instancia de `Person`

Podemos usar `Logger` para decorar funcoes "soltas" tambem

In [120]:
@Logger
def say_bye():
    pass

In [121]:
say_bye()

Log: say_bye called.


In [122]:
attr_list = [attr for attr in dir(say_bye)]
styled_list = [Text(attr_item, style="red") if attr_item == "__get__" else Text(attr_item, style="blue") for attr_item in attr_list]
columns = Columns(styled_list)
console.print(columns)

__call__   __class__  __delattr__      __dict__    __dir__    __doc__       __eq__            __format__ 
__ge__     __get__    __getattribute__ __gt__      __hash__   __init__      __init_subclass__ __le__     
__lt__     __module__ __ne__           __new__     __reduce__ __reduce_ex__ __repr__          __setattr__
__sizeof__ __str__    __subclasshook__ __weakref__ fn        

Como podemos ver, o metodo `__get__` sequer e chamado. 

Podemos verificar se e possivel usar o decorador `Logger` com `staticmethod` e `classmethod`

Lembrete importante:

Da forma que foi implementado, `Logger` nao faz o desempacotamento desse tipo de metodo sozinha. 

Assim, precisamos decorar esses metodos `staticmethod` e `classmethod` empilhando os decoradores adequadamente. 

In [123]:
class Person:
    @classmethod
    @Logger
    def cls_method(cls):
        print('class method called...')
        
    @staticmethod
    @Logger
    def static_method():
        print('static method called...')

In [124]:
Person.cls_method()

__get__ called: self=<__main__.Logger object at 0x000002B61BD54FA0>, instance=<class '__main__.Person'>
	returning self as a method bound to instance
Log: cls_method called.
class method called...


In [125]:
Person.static_method()

Log: static_method called.
static method called...
